# 03 - MLP em PyTorch para Churn

Pipeline robusto de Rede Neural Multicamadas (MLP) em PyTorch para previsão de churn, com automação e rastreabilidade no MLflow.

**Sumário:**
1. Setup e bibliotecas
2. Preparação dos dados
3. Definição da arquitetura MLP (com BatchNorm e Dropout)
4. DataLoader e configuração de treinamento
5. Treinamento com mini-batches e early stopping
6. Avaliação completa (Accuracy, F1, ROC-AUC, PR-AUC, Confusion Matrix)
7. Comparação com baselines
8. Análise de trade-off de custo por threshold
9. Registro no MLflow

## 1) Setup e bibliotecas

Importação das principais bibliotecas utilizadas:
- PyTorch para modelagem e treinamento
- Numpy e pandas para manipulação de dados
- Matplotlib para visualização
- Configuração do device (GPU/CPU)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from joblib import load

import mlflow
import mlflow.pytorch
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device em uso: {device}')

# Constantes de negócio (mesmas do notebook de baselines)
V_RETIDO = 500.0   # valor retido por churn evitado (R$)
C_ACAO   = 50.0    # custo de abordagem por cliente (R$)

## 2) Preparação dos dados
Reaproveitamento do pipeline de preprocessamento do baseline para garantir comparabilidade entre modelos.

- Carrega os splits já tratados (X_train, X_test, y_train, y_test) exportados do notebook baseline.
- Converte todos os dados para arrays float32, garantindo compatibilidade com PyTorch.
- Exporta os dados como tensores PyTorch prontos para uso no modelo.

In [ ]:
splits_path = Path('../data/processed/baseline_splits_arrays.joblib')

if not splits_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {splits_path}\n"
        "Execute o notebook 02_baselines.ipynb primeiro para gerar os splits."
    )

X_train, X_test, y_train, y_test = load(splits_path)

print(f"X_train: {X_train.shape} | dtype: {X_train.dtype}")
print(f"X_test : {X_test.shape}  | dtype: {X_test.dtype}")
print(f"y_train: {y_train.shape} | churn rate: {y_train.mean():.3f}")
print(f"y_test : {y_test.shape}  | churn rate: {y_test.mean():.3f}")

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32).to(device)

print(f"Tensores criados — X_train: {X_train_tensor.shape} | X_test: {X_test_tensor.shape}")

## 3) Definição da arquitetura MLP

MLP com duas camadas ocultas, BatchNorm e Dropout para regularização.

- **BatchNorm1d**: estabiliza o treinamento e acelera a convergência
- **Dropout**: reduz overfitting ao desativar neurônios aleatoriamente durante o treino
- **Saída sem ativação**: usamos `BCEWithLogitsLoss`, que combina Sigmoid + BCE de forma numericamente estável

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
            # Sem Sigmoid: BCEWithLogitsLoss aplica internamente (mais estável)
        )

    def forward(self, x):
        return self.net(x)


input_dim = X_train_tensor.shape[1]
model = MLP(input_dim).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParâmetros treináveis: {total_params:,}")

## 4) DataLoader e configuração de treinamento

Configura mini-batches e reserva 10% do treino como validação interna para early stopping.

- **Batching**: divide o treino em lotes, reduzindo uso de memória e permitindo atualizações mais frequentes
- **Early stopping**: interrompe o treino quando a val_loss para de melhorar (evita overfitting)
- **ReduceLROnPlateau**: reduz o LR automaticamente se a val_loss estagna

In [ ]:
# Hiperparâmetros de treinamento
BATCH_SIZE = 512
LR         = 1e-3
N_EPOCHS   = 100
PATIENCE   = 10    # early stopping: epochs sem melhora na val_loss
HIDDEN1    = 128
HIDDEN2    = 64
DROPOUT    = 0.3

# DataLoaders
full_train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_size      = int(0.1 * len(full_train_ds))
train_size    = len(full_train_ds) - val_size
train_ds, val_ds = random_split(
    full_train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Treino: {train_size:,} amostras | Validação: {val_size:,} | Teste: {len(y_test_tensor):,}")
print(f"Batches por epoch: {len(train_loader)}")

## 5) Treinamento com mini-batches e early stopping

In [ ]:
model     = MLP(input_dim, hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

best_val_loss    = float('inf')
best_state       = None
patience_counter = 0
train_losses, val_losses = [], []

for epoch in range(N_EPOCHS):
    # --- treino ---
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch).squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    train_loss = float(np.mean(batch_losses))

    # --- validação ---
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            loss = criterion(model(X_batch).squeeze(), y_batch)
            val_batch_losses.append(loss.item())
    val_loss = float(np.mean(val_batch_losses))

    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # --- early stopping ---
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_state       = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{N_EPOCHS} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping na epoch {epoch+1} — melhor val_loss: {best_val_loss:.4f}")
        break

model.load_state_dict(best_state)
print(f"\nModelo restaurado ao estado da melhor epoch (val_loss: {best_val_loss:.4f})")

# Curva de loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='train_loss')
ax.plot(val_losses,   label='val_loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.set_title('Curva de Treinamento')
ax.legend()
plt.tight_layout()
plt.show()

## 6) Avaliação completa no conjunto de teste

Métricas avaliadas: Accuracy, F1, ROC-AUC, PR-AUC, Confusion Matrix.

In [ ]:
model.eval()
with torch.no_grad():
    logits_test = model(X_test_tensor).squeeze()
    probs_test  = torch.sigmoid(logits_test).cpu().numpy()
    preds_test  = (probs_test >= 0.5).astype(int)
    y_true      = y_test_tensor.cpu().numpy().astype(int)

acc     = accuracy_score(y_true, preds_test)
f1      = f1_score(y_true, preds_test)
roc_auc = roc_auc_score(y_true, probs_test)
pr_auc  = average_precision_score(y_true, probs_test)
cm      = confusion_matrix(y_true, preds_test)

tp = int(((y_true == 1) & (preds_test == 1)).sum())
fp = int(((y_true == 0) & (preds_test == 1)).sum())
valor_liquido = tp * V_RETIDO - (tp + fp) * C_ACAO

print(f"Accuracy : {acc:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")
print(f"Valor Líquido: R$ {valor_liquido:,.0f} (TP={tp}, FP={fp})")
print(f"\n{classification_report(y_true, preds_test, target_names=['Nao Churn', 'Churn'])}")

# Matriz de confusão
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['Real 0', 'Real 1'], ax=ax)
ax.set_title('Matriz de Confusão — MLP PyTorch')
plt.tight_layout()
plt.show()

## 7) Comparação com baselines

Compara o MLP com os resultados do `DummyClassifier` e `LogisticRegression` do notebook anterior.

In [ ]:
# Resultados dos baselines (do notebook 02_baselines.ipynb)
baselines = {
    'dummy_stratified': {'accuracy': 0.5294, 'f1': 0.3937, 'roc_auc': 0.5046,  'pr_auc': 0.3959,  'valor_liquido': 561850},
    'log_reg (C=0.1)':  {'accuracy': 0.8069, 'f1': 0.7425, 'roc_auc': 0.8784,  'pr_auc': 0.8480,  'valor_liquido': 1190800},
    'mlp_pytorch':      {'accuracy': acc,     'f1': f1,     'roc_auc': roc_auc, 'pr_auc': pr_auc, 'valor_liquido': valor_liquido},
}

comparison_df = pd.DataFrame(baselines).T
comparison_df.index.name = 'modelo'
print("=" * 70)
print("Comparação de modelos")
print("=" * 70)
print(comparison_df.to_string(float_format='{:.4f}'.format))

# Ganho do MLP sobre log_reg
delta_roc  = acc - 0.8784
delta_f1   = f1  - 0.7425
delta_vl   = valor_liquido - 1190800
print(f"\nGanho MLP vs. log_reg:")
print(f"  ROC-AUC: {delta_roc:+.4f}")
print(f"  F1     : {delta_f1:+.4f}")
print(f"  Valor  : R$ {delta_vl:+,.0f}")

## 8) Análise de trade-off de custo por threshold

Varre thresholds de 0.10 a 0.90 para identificar o ponto de máximo valor líquido da campanha de retenção.

- **FP** (falso positivo) = cliente abordado desnecessariamente → custo `C_ACAO`
- **FN** (falso negativo) = churn não capturado → perda de `V_RETIDO`
- **Valor líquido** = TP × V_RETIDO − (TP + FP) × C_ACAO

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
thr_results = []

for thr in thresholds:
    p = (probs_test >= thr).astype(int)
    tp_t  = int(((y_true == 1) & (p == 1)).sum())
    fp_t  = int(((y_true == 0) & (p == 1)).sum())
    fn_t  = int(((y_true == 1) & (p == 0)).sum())
    vl    = tp_t * V_RETIDO - (tp_t + fp_t) * C_ACAO
    f1_t  = f1_score(y_true, p, zero_division=0)
    thr_results.append({
        'threshold': round(thr, 2),
        'tp': tp_t, 'fp': fp_t, 'fn': fn_t,
        'clientes_abordados': tp_t + fp_t,
        'f1': round(f1_t, 4),
        'valor_liquido': round(vl, 0),
    })

thr_df   = pd.DataFrame(thr_results)
best_idx = thr_df['valor_liquido'].idxmax()
best_thr = thr_df.loc[best_idx, 'threshold']

print(thr_df.to_string(index=False))
print(f"\nMelhor threshold por valor líquido: {best_thr:.2f}")
print(f"  Valor líquido: R$ {thr_df.loc[best_idx, 'valor_liquido']:,.0f}")
print(f"  Clientes abordados: {int(thr_df.loc[best_idx, 'clientes_abordados']):,}")

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(thr_df['threshold'], thr_df['valor_liquido'], marker='o', color='steelblue')
axes[0].axvline(best_thr, color='red', linestyle='--', label=f'best thr={best_thr:.2f}')
axes[0].set_title('Valor Líquido por Threshold')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('R$')
axes[0].legend()

axes[1].plot(thr_df['threshold'], thr_df['f1'], marker='o', color='darkorange', label='F1')
axes[1].plot(thr_df['threshold'], thr_df['clientes_abordados'] / len(y_true),
             marker='s', linestyle='--', color='gray', label='% Abordados')
axes[1].set_title('F1 e Cobertura por Threshold')
axes[1].set_xlabel('Threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9) Registro no MLflow

In [ ]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("churn-mlp-pytorch")

with mlflow.start_run(run_name="mlp_pytorch_v1"):
    # Parâmetros
    mlflow.log_params({
        "model":                  "mlp_pytorch",
        "architecture":           f"{input_dim}->{HIDDEN1}->{HIDDEN2}->1",
        "hidden1":                HIDDEN1,
        "hidden2":                HIDDEN2,
        "dropout":                DROPOUT,
        "batch_size":             BATCH_SIZE,
        "learning_rate":          LR,
        "optimizer":              "Adam",
        "weight_decay":           1e-4,
        "loss_fn":                "BCEWithLogitsLoss",
        "early_stopping_patience": PATIENCE,
        "best_threshold":         float(best_thr),
        "epochs_trained":         len(train_losses),
    })

    # Métricas técnicas
    mlflow.log_metrics({
        "accuracy":            float(acc),
        "f1":                  float(f1),
        "roc_auc":             float(roc_auc),
        "pr_auc":              float(pr_auc),
        "best_val_loss":       float(best_val_loss),
    })

    # Métricas de negócio
    mlflow.log_metrics({
        "tp":                  float(tp),
        "fp":                  float(fp),
        "clientes_abordados":  float(tp + fp),
        "valor_liquido":       float(valor_liquido),
        "valor_por_cliente":   float(valor_liquido / len(y_true)),
    })

    mlflow.pytorch.log_model(model, "mlp_model")

    run_id = mlflow.active_run().info.run_id
    print(f"Experimento registrado — run_id: {run_id}")
    print(f"Acesse: mlflow ui --backend-store-uri ./mlruns  (ou make mlflow-up)")